Goal for this file: test out widget functions of shiny to build a graph with a dropdown menu for selecting the year

In [1]:
# import packages
# Import packages
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import platform
from pathlib import Path
import geopandas as gpd
import calendar
import pygris

# Set sns theme
sns.set_theme(style='whitegrid')

In [2]:
# access fire data via function
def read_fires(year):
    #  SET PATH TO DATA FOLDER ON BOX DRIVE
    if platform.system() == 'Windows':
        box_path = Path.home() / 'Box'
    # if mac:
    else:
        box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

    #Confirm that the box path is valid
    if box_path.exists():
        print(f'Box folder found at {box_path}')
    else:
        print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

    # create path object that goes to the specific folder on Box drive
    # uploading wetlands that are bigger than a hectare
    fire_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'fire25_1.gdb'
    print(fire_path.exists())

    fire_shp = gpd.read_file(fire_path, layer='firep25_1')

    # FILTER BY DESIRED YEAR
    fire_yr = fire_shp[fire_shp['YEAR_'] == year]

    # drop rows with null values in geometry column
    fire_yr = fire_yr.dropna(subset=['geometry'])

    # add column for month of alarm date
    # add a column for month of alarm date
    # add a column for month of alarm date
    fire_yr['Alarm Month'] = fire_yr['ALARM_DATE'].dt.month

    fire_yr['Alarm Month Abb'] = pd.Categorical(
        values = fire_yr['ALARM_DATE'].dt.month_name().str[:3],
        categories=calendar.month_abbr[1:]
    )

    return fire_yr

In [3]:
# access wetland data
# set a path to the main box folder based on machine's os
# if windows: 
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
# if mac:
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#Confirm that the box path is valid
if box_path.exists():
    print(f'Box folder found at {box_path}')
else:
    print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

# create path object that goes to the specific folder on Box drive
wetlands_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'wetlands_subset' / 'wetlands_subset.geojson'
print(wetlands_path.exists())

# read in the wetlands geojson file, this will take a WHILE - 10 mins 1st try 
wetlands = gpd.read_file(wetlands_path, engine="pyogrio")

Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True


In [4]:
# DISTANCE TO NEAREST FIRE FUNCTION
def dist_fire(wetlands_shp, fire_shp):
    # convert to same CRS
    wetlands_shp = wetlands_shp.to_crs(epsg=26910)
    fire_shp = fire_shp.to_crs(epsg=26910)
    
    
    # Nearest Neighbor
    nearest_fires = gpd.sjoin_nearest(
        wetlands_shp, fire_shp, how='left', distance_col='Dist_Closest_m'
    )

    # convert distance column from meters to kilometers
    nearest_fires['Dist_Closest_km'] = nearest_fires['Dist_Closest_m'] / 1000

    # add information to a table called fire_stats
    fire_stats = (nearest_fires.groupby('C System')['Dist_Closest_km'].agg(
        Avg_Dist_Fire_km="mean",
        Std_Dist_Fire_km="std",
        Median_Dist_Fire_km="median",
        Range_Dist_Fire_km=lambda x: x.max() - x.min(),
        IQR_Dist_Fire_km=lambda x: x.quantile(0.75) - x.quantile(0.25),
        ).reset_index()
    )

    return nearest_fires, fire_stats
# outputs are a shapefile and the fire_stats table

In [ ]:
# function for graphing
def plot_nutrients(start_date = "1995-01-01",
                   end_date = "1999-12-31",
                   nutrient = "Total Nitrogen",
                   depth_zones = [1,2,3],
                   lake_names = ['Peter Lake', 'Paul Lake']):
    # Subset the data
    df_subset = df_nutrients.loc[
        (df_nutrients['sampledate'] >= start_date) &  # Start date
        (df_nutrients['sampledate'] <= end_date) &    # End date
        (df_nutrients['nutrient'] == nutrient) &  # Nutrient
        (df_nutrients['Depth zone']).isin(depth_zones) & # Depth IDs
        (df_nutrients['Lake name']).isin(lake_names), # Lake Names
    ]

    #Create a plot of selected nutrients by sampledate, with markers colored by depth_id using the inverse mako colormap and different shapes for lakename
    the_plot = sns.relplot(
        data = df_subset,
        x='sampledate',
        y='ug',
        hue='Depth zone',
        style='Lake name',  markers=['d','o'],
        palette='magma',
        s=20,
        alpha=0.7,
    )

    #Set plot title and axis labels
    the_plot.set_axis_labels('Sample Date', nutrient)
    the_plot.set(title=f'{nutrient}')
    the_plot.set_xticklabels(rotation=45, ha='right')

    #Clean up the legend
    sns.move_legend(
        the_plot, 
        loc = 'center right',
        frameon = True,
    )

    #return the plot
    return the_plot

In [ ]:
#Create a dropdown widget for years
year_list = [2020, 2021, 2022, 2023, 2024, 2025]
year_dropdown = widgets.Dropdown(
    options=year_list,
    value=year_list[0],
    description='Year:',
)